# Separation benchmark AudioMixer

This notebook shows how to create one deterministic speech/music mixture. Replace the two input paths with existing audio files on your machine. The mixer leaves those inputs untouched and writes three new floating-point WAV files to the output directory.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from src.benchmark.separation import AudioMixer
from src.utils.AudioClass import Audio

In [ ]:
speech_path = Path("/path/to/speech.wav")
music_path = Path("/path/to/music.wav")
output_dir = Path("output/benchmark_mix_example")

speech = Audio.from_file(speech_path, source_id="example-speech")
music = Audio.from_file(music_path, source_id="example-music")

The speech duration determines the output duration. Shorter music loops to cover the full speech; longer music is cropped. The seed selects a repeatable starting point, and positive SMR values make speech louder relative to music.

In [ ]:
mixer = AudioMixer(
    sample_rate=44_100,
    channels=2,
    peak_ceiling_dbfs=-1.0,
)

result = mixer.mix(
    speech,
    music,
    target_smr_db=0.0,
    seed=12_345,
    output_dir=output_dir,
)

In [ ]:
print("Speech reference:", result.speech_reference.path)
print("Music reference: ", result.music_reference.path)
print("Mixture:        ", result.mixture.path)
print("Parameters:     ", result.parameters)

The three returned `Audio` objects point to `speech_reference.wav`, `music_reference.wav`, and `mixture.wav`. The two references are the scaled components actually used in the mixture. `result.parameters.music_start_sample` records the crop, and `realized_rms_smr_db` reports the measured RMS ratio.